# Data preprocessing — California single-family price prediction

Goal: predict close price for a California single-family address at prediction time, including homes that are off market. Use property characteristics and location that can be obtained for that address. Sold-record attributes must be checked against the eventual prediction-time data source.

Deliverables: this notebook and `data/cleaned_single_family_sales.csv`. The cleaned table retains original units and missing features. Training-only preprocessing handles imputation and scaling. Export does not depend on choosing a model or training window.

Reserve the most recent available sale month for testing. Tune training-window length using earlier validation months, never the test month. Report gaps and whether the newest month is incomplete.

Input context: six historical monthly CRMLS exports, November 2025 through April 2026. The provider background does not establish whether repeated keys represent corrections or distinct sale events.


## Setup and configuration

Run cells in order. Configuration declares the current rules; inputs remain unchanged in `raw`.


In [71]:
# Standard-library imports keep configuration and run metadata portable.
import hashlib
import json
import platform
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Keep cleaning/model choices together so each run uses an explicit policy.
NUMERIC_CANDIDATES = [
    "LivingArea", "LotSizeSquareFeet", "BedroomsTotal", "BathroomsTotalInteger",
    "GarageSpaces", "ParkingTotal", "Stories", "YearBuilt", "Latitude", "Longitude",
]
CATEGORICAL_CANDIDATES = ["City", "PostalCode"]
REQUIRED_COLUMNS = {
    "PropertyType", "PropertySubType", "StateOrProvince",
    "ClosePrice", "CloseDate", "ListingKeyNumeric",
}
TEXT_COLUMNS = {
    "ListingKeyNumeric": "string", "ListingKey": "string",
    "ListingId": "string", "PostalCode": "string",
}
COORDINATE_BOUNDS = {"Latitude": (-90, 90), "Longitude": (-180, 180)}
CANDIDATE_WINDOWS = [1, 3, 6, 12]
# With April reserved for testing, use February and March as shared folds.
VALIDATION_OFFSETS = [2, 1]
RIDGE_ALPHA = 1.0
# Conservative baseline screening assumptions, not physical or CRMLS limits.
MEASUREMENT_SCREENING = {
    "minimum_positive_lot_sqft": 1.0,
    "suspected_double_conversion_minimum_area": 1000.0,
    "garage_review_limit": 20.0,
    "parking_review_limit": 50.0,
}
SQFT_PER_ACRE = 43560.0
DUPLICATE_POLICY = "quarantine_all_conflicting_keys"
# Manually reviewed sale-price anomalies. No numeric corrections are inferred.
PRICE_REVIEW_REASONS = {
    "1144223437": "$1.75 close price versus $1,749,000 list price",
    "1110035161": "$8,300 close price versus $2,000,000 list price",
    "1138900104": "$31,000 close price versus $310,000 list price",
    "1126691917": "$37,000 close price versus $385,000 list price",
    "1149697906": "$796,000,000 close price versus $824,900 list price",
    "1150712418": "$768,500,000 close price versus $750,000 list price",
    "1126046881": "$740,000,000 close price versus $735,000 list price",
    "1148201765": "$664,250,000 close price versus $750,000 list price",
    "1153305774": "$645,000,000 close price versus $645,000 list price",
    "1135757473": "$472,000,000 close price versus $479,000 list price",
    "1111473680": "$199,947,850 close price versus $1,499,000 list price",
}

# Resolve paths from either the project root or the notebooks directory.
project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "notebooks" / "01_exploration.ipynb").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the kernel in the project root or notebooks directory.")
data_dir = project_root / "data"


def file_sha256(path):
    """Fingerprint an input or artifact without loading the whole file into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def audit_step(step, before, after):
    """Record actual row removals; validity checks may count overlapping issues."""
    row_audit.append({
        "step": step, "rows_remaining": int(after), "rows_removed": int(before - after),
    })


## Import monthly source files


In [72]:
# Sort file paths to make import order deterministic. Exclude generated CSVs.
files = sorted(
    path for path in data_dir.glob("*")
    if path.is_file() and path.suffix.lower() == ".csv"
    and path.name.lower().startswith("crmlssold")
)
if not files:
    raise FileNotFoundError(f"No CRMLSSold CSVs found in {data_dir}")

# Validate each month's schema, rather than hiding omissions in concatenation.
frames = []
input_manifest = []
for path in files:
    frame = pd.read_csv(path, low_memory=False, dtype=TEXT_COLUMNS)
    missing_columns = REQUIRED_COLUMNS - set(frame.columns)
    if missing_columns:
        raise ValueError(f"{path.name}: required columns missing: {sorted(missing_columns)}")
    frames.append(frame.assign(source_file=path.name))
    input_manifest.append({
        "file": path.name, "rows": len(frame), "sha256": file_sha256(path),
    })
raw = pd.concat(frames, ignore_index=True)
del frames, frame  # Release references to the individual imports after concatenation.
print(f"Imported {len(raw):,} rows from {len(files)} files")
print("\nInput files — row counts and content fingerprints:")
display(pd.DataFrame(input_manifest))


Imported 122,756 rows from 6 files

Input files — row counts and content fingerprints:


,file,rows,sha256
0,CRMLSSold202511.csv,19088,c31e619e5d303a17c9058779ea1a25e4d49c1459f8a0ee...
1,CRMLSSold202512.csv,20538,f0b6985e04e4203a859b9993a8186a1d7f4b191d057b57...
2,CRMLSSold202601.csv,16487,7cb4d45b59cd94415a72855019de530aa8d8bc4755f4ca...
3,CRMLSSold202602.csv,19010,0f737b261c728c6b76ec9a9bacd19d5045110a6c462cd7...
4,CRMLSSold202603.csv,23372,89add885c9f6e11fd84269122a809c4b2e47e262e948ae...
5,CRMLSSold202604.csv,24261,be290be4fe135ef6be4ab645ed0e87e48ef4f1cbfc0461...


## 1. California scope, target validity, and duplicates

Keep residential single-family sales with a normalized state value of `CA` or `California`. Exclude and count missing or other states. A California state label establishes scope; it does not independently verify coordinates.

Exclude nonpositive/nonfinite prices, unparseable sale dates, and missing/blank listing keys. Remove exact repeats ignoring source provenance. **Set aside every record belonging to a conflicting listing key** in a separate review table. Do not infer that the newest monthly file contains a correction. Export the review records separately so they can be verified and restored later.

Review of the supplied outputs found 39 conflicting keys covering 79 records: all have different close dates and 12 have different close prices. This is an observation from the current inputs, not a fixed row-count requirement. Exclusion may affect sample coverage; record actual counts on every run.

Validity counts are measured before exclusion and can overlap. The audit reports actual row removals. Positive price extremes remain for review; passing these checks does not prove every value is accurate.


In [73]:
# Reset the audit when this cell is rerun; always start from the unchanged import.
row_audit = [{"step": "imported", "rows_remaining": len(raw), "rows_removed": 0}]

# 1a. Scope: California residential single-family records.
state = raw["StateOrProvince"].astype("string").str.strip().str.upper()
print("\nState / province counts before California filtering (includes missing values):")
display(state.value_counts(dropna=False).rename_axis("State / province").rename("Row count").to_frame())
cleaned = raw.loc[
    raw["PropertyType"].eq("Residential") &
    raw["PropertySubType"].eq("SingleFamilyResidence") &
    state.isin(["CA", "CALIFORNIA"])
].copy()
audit_step("California residential single-family scope", len(raw), len(cleaned))
cleaned["StateOrProvince"] = "CA"
cleaned["ListingKeyNumeric"] = cleaned["ListingKeyNumeric"].astype("string").str.strip().replace("", pd.NA)
# 1b. Validate target, sale date, and listing identity before excluding rows.
cleaned["ClosePrice"] = pd.to_numeric(cleaned["ClosePrice"], errors="coerce")
cleaned["CloseDate"] = pd.to_datetime(cleaned["CloseDate"], errors="coerce")
print("Rows with missing, nonnumeric, nonfinite, or nonpositive close prices:", (~(cleaned["ClosePrice"].gt(0) & np.isfinite(cleaned["ClosePrice"]))).sum())
print("Rows with missing or unparseable close dates:", cleaned["CloseDate"].isna().sum())
print("Rows with missing or blank listing keys:", cleaned["ListingKeyNumeric"].isna().sum())
before = len(cleaned)
cleaned = cleaned.loc[
    cleaned["ClosePrice"].gt(0) & np.isfinite(cleaned["ClosePrice"]) &
    cleaned["CloseDate"].notna() & cleaned["ListingKeyNumeric"].notna()
].copy()
audit_step("valid price, date, and listing key", before, len(cleaned))
# 1c. Remove exact repeats, then inspect differing records sharing a key.
record_columns = [c for c in cleaned if c != "source_file"]
before = len(cleaned)
cleaned = cleaned.drop_duplicates(subset=record_columns)
audit_step("exact duplicates", before, len(cleaned))
print("Exact duplicate rows removed (source file ignored):", before - len(cleaned))
conflicts = cleaned.loc[cleaned.duplicated("ListingKeyNumeric", keep=False)]
if not conflicts.empty:
    print("Listing keys with conflicting records:", conflicts["ListingKeyNumeric"].nunique())
    # Match current model candidates; include sale identity and scope fields.
    review_columns = [
        col for col in ["ClosePrice", "CloseDate", "PropertyType", "PropertySubType",
                       "StateOrProvince", *NUMERIC_CANDIDATES, *CATEGORICAL_CANDIDATES]
        if col in cleaned
    ]
    conflict_summary_rows = []
    important_difference_rows = []
    for key, records in conflicts.groupby("ListingKeyNumeric", sort=True):
        changed = [col for col in record_columns
                   if records[col].nunique(dropna=False) > 1]
        important_changed = [col for col in review_columns if col in changed]
        conflict_summary_rows.append({
            "Listing key": key,
            "Record count": len(records),
            "Source files": ", ".join(sorted(records["source_file"].unique())),
            "Changed columns": ", ".join(changed),
            "Changed target / date / scope / predictors": ", ".join(important_changed) or "None",
        })
        for col in important_changed:
            for _, record in records.iterrows():
                important_difference_rows.append({
                    "Listing key": key,
                    "Field": col,
                    "Source file": record["source_file"],
                    "Value": str(record[col]) if pd.notna(record[col]) else "<missing>",
                })
    conflict_summary = pd.DataFrame(conflict_summary_rows)
    important_differences = pd.DataFrame(important_difference_rows)
    print("\nDuplicate review — all conflicting keys and the columns that differ:")
    with pd.option_context("display.max_rows", None, "display.max_columns", None,
                           "display.max_colwidth", None):
        display(conflict_summary)
        print("\nDifferences affecting the target, sale date, scope, or current model candidates:")
        if important_differences.empty:
            print("None. All conflicts are confined to fields outside the current model inputs and sale identity checks.")
        else:
            display(important_differences)
    # Every version is retained separately for review; no sale version is assumed correct.
    print(f"\nSetting aside {len(conflicts):,} records across "
          f"{conflicts['ListingKeyNumeric'].nunique():,} conflicting keys.")

# Keep a separate review table even when a future run has no conflicting records.
duplicate_review = conflicts.copy()
duplicate_review["review_reason"] = "Different records share the same listing key"
before = len(cleaned)
cleaned = cleaned.loc[~cleaned["ListingKeyNumeric"].isin(conflicts["ListingKeyNumeric"])].copy()
audit_step("conflicting listing keys set aside for review", before, len(cleaned))
cleaned = cleaned.reset_index(drop=True)
assert not cleaned.empty, "No usable rows remain."

print("\nCleaning audit — rows remaining and removed at each step:")
display(pd.DataFrame(row_audit).rename(columns={
    "step": "Cleaning step", "rows_remaining": "Rows remaining", "rows_removed": "Rows removed"
}))
print("\nClose-price summary after scope, validity, and duplicate checks (original currency units):")
display(cleaned["ClosePrice"].describe(percentiles=[0.01, 0.5, 0.99])
        .rename_axis("Statistic").rename("Close price").to_frame())



State / province counts before California filtering (includes missing values):


,Row count
State / province,
CA,122745
AZ,3
FL,3
OR,1
HI,1
NM,1
NY,1
OH,1


Rows with missing, nonnumeric, nonfinite, or nonpositive close prices: 0
Rows with missing or unparseable close dates: 0
Rows with missing or blank listing keys: 0
Exact duplicate rows removed (source file ignored): 0
Listing keys with conflicting records: 39

Duplicate review — all conflicting keys and the columns that differ:


,Listing key,Record count,Source files,Changed columns,Changed target / date / scope / predictors
0,1107335546,2,"CRMLSSold202601.csv, CRMLSSold202604.csv","BuyerAgentAOR, CloseDate, DaysOnMarket, BuyerOfficeName, BuyerAgentMlsId, BuyerAgentFirstName, BuyerAgentLastName, ContractStatusChangeDate, PurchaseContractDate, AssociationFee, OriginatingSystemName, OriginatingSystemSubName",CloseDate
1,1108963864,2,"CRMLSSold202601.csv, CRMLSSold202604.csv","BuyerAgentAOR, CloseDate, ContractStatusChangeDate, OriginatingSystemName, OriginatingSystemSubName",CloseDate
2,1110678316,2,"CRMLSSold202511.csv, CRMLSSold202512.csv","CloseDate, ClosePrice, BuyerOfficeName, BuyerAgentMlsId, BuyerAgentFirstName, BuyerAgentLastName, BuyerOfficeAOR, ContractStatusChangeDate","ClosePrice, CloseDate"
3,1113067986,2,"CRMLSSold202511.csv, CRMLSSold202512.csv","CloseDate, ContractStatusChangeDate",CloseDate
4,1114405045,2,"CRMLSSold202511.csv, CRMLSSold202603.csv","BuyerAgentAOR, CloseDate, Latitude, BuyerAgentFirstName, BuyerAgentLastName, ContractStatusChangeDate, OriginatingSystemName, OriginatingSystemSubName","CloseDate, Latitude"
5,1118677354,2,"CRMLSSold202511.csv, CRMLSSold202512.csv","CloseDate, ContractStatusChangeDate",CloseDate
6,1118835836,2,"CRMLSSold202602.csv, CRMLSSold202603.csv","CloseDate, ContractStatusChangeDate",CloseDate
7,1118859284,2,"CRMLSSold202601.csv, CRMLSSold202602.csv","BuyerAgentAOR, CloseDate, ClosePrice, ContractStatusChangeDate, OriginatingSystemName, OriginatingSystemSubName","ClosePrice, CloseDate"
8,1119101932,2,"CRMLSSold202511.csv, CRMLSSold202512.csv","CloseDate, ContractStatusChangeDate",CloseDate
9,1119599421,2,"CRMLSSold202512.csv, CRMLSSold202602.csv","BuyerAgentAOR, CloseDate, ClosePrice, DaysOnMarket, BuyerOfficeName, BuyerAgentMlsId, BuyerAgentFirstName, BuyerAgentLastName, BuyerOfficeAOR, ContractStatusChangeDate, PurchaseContractDate, OriginatingSystemName, OriginatingSystemSubName","ClosePrice, CloseDate"



Differences affecting the target, sale date, scope, or current model candidates:


,Listing key,Field,Source file,Value
0,1107335546,CloseDate,CRMLSSold202601.csv,2026-01-26 00:00:00
1,1107335546,CloseDate,CRMLSSold202604.csv,2026-04-16 00:00:00
2,1108963864,CloseDate,CRMLSSold202601.csv,2026-01-09 00:00:00
3,1108963864,CloseDate,CRMLSSold202604.csv,2026-04-17 00:00:00
4,1110678316,ClosePrice,CRMLSSold202511.csv,290000.0
5,1110678316,ClosePrice,CRMLSSold202512.csv,299000.0
6,1110678316,CloseDate,CRMLSSold202511.csv,2025-11-07 00:00:00
7,1110678316,CloseDate,CRMLSSold202512.csv,2025-12-23 00:00:00
8,1113067986,CloseDate,CRMLSSold202511.csv,2025-11-18 00:00:00
9,1113067986,CloseDate,CRMLSSold202512.csv,2025-12-16 00:00:00



Setting aside 79 records across 39 conflicting keys.

Cleaning audit — rows remaining and removed at each step:


,Cleaning step,Rows remaining,Rows removed
0,imported,122756,0
1,California residential single-family scope,60694,62062
2,"valid price, date, and listing key",60694,0
3,exact duplicates,60694,0
4,conflicting listing keys set aside for review,60615,79



Close-price summary after scope, validity, and duplicate checks (original currency units):


,Close price
Statistic,
count,6.061500e+04
mean,1.320965e+06
std,7.039950e+06
min,1.750000e+00
1%,2.300000e+05
50%,8.800000e+05
99%,6.573600e+06
max,7.960000e+08


### Section 1 findings and remaining review

The reviewed run imported **122,756 records**, retained **60,694 California residential single-family records**, and set aside **79 records across 39 conflicting listing keys**, leaving **60,615 records** for feature preprocessing. The scoped records had no invalid/nonpositive prices, unparseable sale dates, missing listing keys, or exact duplicates under the checks above. The 62,062 scope exclusions combine property-type, subtype, and state criteria; they are not all out-of-state records.

The retained median close price was **$880,000**, with the 1st and 99th percentiles at **$230,000** and **$6,573,600**. However, the minimum of **$1.75** and maximum of **$796 million** require investigation. A positive price passes the validity check without establishing that it is a plausible single-property sale. Do not automatically remove all observations outside the percentile range or assume these extremes are valid.

**Next:** proceed to section 2 to inspect feature availability, missingness, and impossible property values. Review the extreme-price records and geographic inconsistencies before treating the CSV as a finalized cleaning deliverable. Conflicting-key records remain available for separate verification.

These figures summarize the output reviewed during this session. After changing inputs or cleaning rules, refresh this narrative from the current audit and descriptive statistics.


## 2. Feature selection and missing-value plan

Initial candidates are property characteristics and location. Confirm that the eventual address lookup supplies these fields at prediction time, including for off-market homes. Listing prices, days on market, buyer details, and sale status are not predictors. Exclude identifiers, provenance, target, final status, and sale date from model inputs. Association fees are deferred until billing periods are standardized.

Use median imputation plus missing indicators for numeric fields; use a missing category plus one-hot encoding for categorical fields. Drop features entirely missing in each training fold. Standard scaling suits the Ridge baseline; tree models may not need it. Do not cap legitimate extremes just because plots capped them.

Review geographic flags from exploration before export. Section 1 excludes non-California state labels. Coordinates outside California and plausible extremes still need review; global latitude/longitude bounds only detect impossible coordinates.


In [74]:
# Convert features without learning any values from the full dataset.
numeric_features = [c for c in NUMERIC_CANDIDATES if c in cleaned]
categorical_features = [c for c in CATEGORICAL_CANDIDATES if c in cleaned]
for c in numeric_features:
    cleaned[c] = pd.to_numeric(cleaned[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
for c, bounds in COORDINATE_BOUNDS.items():
    if c in numeric_features:
        cleaned.loc[~cleaned[c].between(*bounds), c] = np.nan
for c in categorical_features:
    cleaned[c] = cleaned[c].map(lambda v: str(v).strip() if pd.notna(v) and str(v).strip() else np.nan)
feature_columns = numeric_features + categorical_features
assert feature_columns, "Select at least one usable feature."
# Counts and percentages make missingness readable without interpreting fractions.
feature_missingness = pd.DataFrame({
    "Missing rows": cleaned[feature_columns].isna().sum(),
    "Available rows": cleaned[feature_columns].notna().sum(),
    "Missing (%)": cleaned[feature_columns].isna().mean().mul(100),
}).sort_values("Missing (%)", ascending=False, kind="stable")
feature_missingness.index.name = "Feature"
print("\nFeature missingness after type conversion and coordinate validity checks:")
display(feature_missingness.style.format({"Missing rows": "{:,.0f}",
                                         "Available rows": "{:,.0f}",
                                         "Missing (%)": "{:.2f}%"}))



Feature missingness after type conversion and coordinate validity checks:


,Missing rows,Available rows,Missing (%)
Feature,,,
Stories,"6,230","54,385",10.28%
GarageSpaces,"2,341","58,274",3.86%
LotSizeSquareFeet,"1,056","59,559",1.74%
Latitude,213,"60,402",0.35%
Longitude,213,"60,402",0.35%
YearBuilt,37,"60,578",0.06%
LivingArea,29,"60,586",0.05%
City,7,"60,608",0.01%
BathroomsTotalInteger,1,"60,614",0.00%


### Review measurements and price extremes

The reviewed missingness output supports retaining all current candidates provisionally: Stories is missing in 10.28% of records, GarageSpaces in 3.86%, and each other feature in less than 2%. Missing values remain in the CSV; training folds supply numeric medians and the categorical missing label.

Missingness alone does not establish validity. The checks below flag nonpositive area/story measurements, negative or fractional counts, and questionable construction years. Flags are for review and do not change values. Zero parking/garage counts can be meaningful, and half-stories can be valid. Inspect the numeric distributions and lowest/highest prices before finalizing rules. Refresh this narrative if the inputs change.


In [75]:
# Review flags do not change values or remove rows. Decide rules after inspection.
validity_checks = {}
for col in ["LivingArea", "LotSizeSquareFeet", "Stories"]:
    if col in numeric_features:
        validity_checks[f"{col}: zero or negative"] = cleaned[col].le(0)
for col in ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal"]:
    if col in numeric_features:
        validity_checks[f"{col}: negative"] = cleaned[col].lt(0)
        validity_checks[f"{col}: fractional count (review)"] = (
            cleaned[col].notna() & cleaned[col].mod(1).ne(0)
        )
# A half-story can be meaningful; do not apply an integer rule to Stories.
if "YearBuilt" in numeric_features:
    validity_checks["YearBuilt: zero or negative"] = cleaned["YearBuilt"].le(0)
    validity_checks["YearBuilt: later than sale year (review)"] = (
        cleaned["YearBuilt"].gt(cleaned["CloseDate"].dt.year)
    )

validity_summary = pd.DataFrame([
    {"Review check": label, "Flagged rows": int(mask.sum())}
    for label, mask in validity_checks.items()
])
print("\nProperty-value review checks (flags can overlap; zeros in count fields are retained):")
display(validity_summary)

# Show actual flagged measurements, rather than a wide table of unrelated fields.
flagged_measurements = pd.concat([
    cleaned.loc[mask, ["ListingKeyNumeric", "CloseDate", *numeric_features]]
           .assign(review_check=label)
    for label, mask in validity_checks.items()
], ignore_index=True) if validity_checks else pd.DataFrame()
if not flagged_measurements.empty:
    print("\nFlagged property measurements — first 30 rows; full table is in flagged_measurements:")
    with pd.option_context("display.max_columns", None):
        display(flagged_measurements.head(30))

print("\nNumeric feature distribution summary — original units, no clipping:")
display(cleaned[numeric_features].describe(percentiles=[0.01, 0.5, 0.99]).T)

# Extreme positive prices passed section 1; inspect them before choosing any exclusions.
price_review_columns = ["ListingKeyNumeric", "CloseDate", "ClosePrice", *feature_columns]
print("\nLowest 10 close prices — retained for review:")
with pd.option_context("display.max_columns", None):
    display(cleaned.nsmallest(10, "ClosePrice")[price_review_columns])
    print("\nHighest 10 close prices — retained for review:")
    display(cleaned.nlargest(10, "ClosePrice")[price_review_columns])



Property-value review checks (flags can overlap; zeros in count fields are retained):


,Review check,Flagged rows
0,LivingArea: zero or negative,18
1,LotSizeSquareFeet: zero or negative,43
2,Stories: zero or negative,0
3,BedroomsTotal: negative,0
4,BedroomsTotal: fractional count (review),0
5,BathroomsTotalInteger: negative,0
6,BathroomsTotalInteger: fractional count (review),0
7,GarageSpaces: negative,0
8,GarageSpaces: fractional count (review),94
9,ParkingTotal: negative,8



Flagged property measurements — first 30 rows; full table is in flagged_measurements:


,ListingKeyNumeric,CloseDate,LivingArea,LotSizeSquareFeet,BedroomsTotal,BathroomsTotalInteger,GarageSpaces,ParkingTotal,Stories,YearBuilt,Latitude,Longitude,review_check
0,1068149606,2025-11-07,0.0,22189.0,6.0,6.0,NaN,4.0,2.0,1923.0,34.072010,-118.405262,LivingArea: zero or negative
1,1149744659,2025-12-22,0.0,6507.0,4.0,3.0,NaN,0.0,1.0,1926.0,34.077172,-118.371810,LivingArea: zero or negative
2,1136715994,2025-12-04,0.0,8206.7,6.0,8.0,2.0,2.0,NaN,2022.0,34.040193,-118.411506,LivingArea: zero or negative
3,1134702110,2025-12-22,0.0,11731.0,3.0,3.0,2.0,4.0,1.0,1967.0,34.129049,-118.489745,LivingArea: zero or negative
4,1150723609,2026-01-23,0.0,4797.0,0.0,0.0,0.0,4.0,1.0,1925.0,32.754698,-117.118260,LivingArea: zero or negative
5,1150066771,2026-01-30,0.0,7389.0,3.0,3.0,2.0,2.0,NaN,2014.0,37.843060,-122.195419,LivingArea: zero or negative
6,1147619191,2026-01-28,0.0,8640.0,3.0,2.0,2.0,5.0,1.0,1935.0,34.148016,-118.410116,LivingArea: zero or negative
7,1147468485,2026-01-16,0.0,5457.0,3.0,1.0,1.0,1.0,1.0,1942.0,37.962140,-122.354761,LivingArea: zero or negative
8,1146102942,2026-01-30,0.0,20364.0,5.0,7.0,NaN,5.0,2.0,1953.0,34.053436,-118.502213,LivingArea: zero or negative
9,1137140936,2026-01-02,0.0,4841.0,4.0,5.0,NaN,0.0,2.0,2016.0,34.078795,-118.378451,LivingArea: zero or negative



Numeric feature distribution summary — original units, no clipping:


,count,mean,std,min,1%,50%,99%,max
LivingArea,60586.0,2050.194177,1.037353e+03,0.000000,740.000000,1819.000000,5683.450000,3.106800e+04
LotSizeSquareFeet,59559.0,542434.659895,2.266338e+07,0.000000,1707.000000,7254.000000,310463.000000,1.938943e+09
BedroomsTotal,60615.0,3.494729,9.702071e-01,0.000000,2.000000,3.000000,6.000000,2.200000e+01
BathroomsTotalInteger,60614.0,2.637542,1.132261e+00,0.000000,1.000000,2.000000,6.000000,2.300000e+01
GarageSpaces,58274.0,1.990974,2.269118e+00,0.000000,0.000000,2.000000,4.000000,4.000000e+02
ParkingTotal,60615.0,2.961726,1.843146e+01,-16.000000,0.000000,2.000000,11.000000,3.921000e+03
Stories,54385.0,1.350703,4.771946e-01,1.000000,1.000000,1.000000,2.000000,2.000000e+00
YearBuilt,60578.0,1976.277873,2.778353e+01,1801.000000,1911.000000,1977.000000,2025.000000,2.026000e+03
Latitude,60402.0,34.698987,1.729955e+00,0.000000,32.667483,34.075392,39.735846,4.378444e+01
Longitude,60402.0,-118.537051,3.583985e+00,-124.175789,-122.444900,-118.006352,-116.241164,1.204327e+02



Lowest 10 close prices — retained for review:


,ListingKeyNumeric,CloseDate,ClosePrice,LivingArea,LotSizeSquareFeet,BedroomsTotal,BathroomsTotalInteger,GarageSpaces,ParkingTotal,Stories,YearBuilt,Latitude,Longitude,City,PostalCode
24745,1144223437,2026-01-30,1.75,3513.0,11326.0,3.0,4.0,3.0,3.0,1.0,2000.0,33.744081,-116.328238,Palm Desert,92211
32838,1147508351,2026-02-04,8000.00,984.0,6200.0,3.0,1.0,0.0,0.0,1.0,1962.0,35.746692,-117.399521,Trona,93562
27485,1110035161,2026-01-17,8300.00,3886.0,19206.0,5.0,5.0,3.0,3.0,2.0,1999.0,34.208003,-118.154204,Altadena,91001
36506,1108190915,2026-02-11,18000.00,1488.0,13068.0,0.0,0.0,0.0,0.0,1.0,1957.0,33.605124,-114.729659,Blythe,92225
43143,1151314793,2026-03-26,28000.00,408.0,54450.0,0.0,0.0,0.0,5.0,1.0,1968.0,34.369680,-116.472823,Landers,92285
7451,1120335789,2025-11-20,29000.00,960.0,43560.0,3.0,1.0,2.0,2.0,1.0,1950.0,34.897592,-116.829557,Yermo,92398
102,1146281620,2025-11-26,29900.00,1198.0,3920.0,1.0,1.0,0.0,2.0,2.0,1953.0,35.400201,-119.466655,Buttonwillow,93206
12175,1145717594,2025-12-10,31000.00,1942.0,20908.8,5.0,3.0,2.0,2.0,1.0,1953.0,35.751441,-117.395198,Trona,93562
25275,1138900104,2026-01-27,31000.00,1309.0,6969.0,4.0,2.0,2.0,2.0,1.0,1955.0,35.370551,-118.948818,Bakersfield,93306
18299,1126691917,2025-12-08,37000.00,1456.0,58370.4,3.0,2.0,2.0,2.0,1.0,1992.0,37.264099,-119.692443,Coarsegold,93614



Highest 10 close prices — retained for review:


,ListingKeyNumeric,CloseDate,ClosePrice,LivingArea,LotSizeSquareFeet,BedroomsTotal,BathroomsTotalInteger,GarageSpaces,ParkingTotal,Stories,YearBuilt,Latitude,Longitude,City,PostalCode
58811,1149697906,2026-04-27,796000000.0,1936.0,14810.40,2.0,3.0,3.0,3.0,1.0,1977.0,33.250628,-117.135871,Valley Center,92082
58371,1150712418,2026-04-09,768500000.0,1444.0,0.12,4.0,2.0,2.0,4.0,1.0,1978.0,32.572879,-117.062559,San Diego,92154
7186,1126046881,2025-11-24,740000000.0,1271.0,5100.00,3.0,2.0,2.0,2.0,1.0,1973.0,32.566454,-117.095029,San Diego,92154
45337,1148201765,2026-03-26,664250000.0,1992.0,0.23,4.0,3.0,2.0,4.0,NaN,1959.0,32.625679,-117.065524,Chula Vista,91911
55080,1153305774,2026-04-22,645000000.0,1104.0,4300.00,2.0,1.0,1.0,2.0,1.0,1972.0,33.185572,-117.302394,Oceanside,92056
25905,1135757473,2026-01-19,472000000.0,1910.0,8200.00,4.0,2.0,2.0,7.0,1.0,2000.0,34.490545,-117.257502,Victorville,92395
9399,1111473680,2025-11-14,199947850.0,1704.0,74393.00,3.0,2.0,2.0,10.0,1.0,1964.0,34.123623,-118.599896,Topanga,90290
50445,1155438031,2026-04-30,60000000.0,15006.0,261796.00,7.0,14.0,NaN,10.0,2.0,NaN,34.430419,-119.646132,Montecito,93108
36733,1155445977,2026-03-16,47000000.0,10286.0,52591.00,7.0,10.0,NaN,0.0,2.0,1937.0,34.090074,-118.407828,Beverly Hills,90210
48359,1157576793,2026-04-14,41250000.0,NaN,39924.00,7.0,10.0,NaN,0.0,2.0,1928.0,34.082565,-118.420343,Beverly Hills,90210


### Investigate possible units and recording errors

The reviewed output contains 18 nonpositive living areas, 43 nonpositive lot areas, 8 negative parking counts, 94 fractional values in each parking/garage field, and 6 construction years later than the sale year. Counts can overlap. The largest lot/parking values, tiny positive lot measurements, and extreme sale prices need source-context review before selecting rules.

Compare lot measurements across available source columns and inspect fractional parking values directly. Missing future construction years may reflect planned completion, so they are not automatically invalid. Listing prices can provide supporting context but cannot establish the correct close price. No multiplying, dividing, rounding, or coordinate sign correction is performed here. These diagnostic fields remain excluded from model inputs.


In [76]:
# Compare suspicious values with related source fields. These are review aids,
# not added model inputs, and no replacement values are inferred automatically.
context_columns = [col for col in [
    "ListingKeyNumeric", "source_file", "CloseDate", "ClosePrice", "City", "PostalCode",
    "LivingArea", "LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea", "LotSizeUnits",
    "GarageSpaces", "ParkingTotal", "YearBuilt", "NewConstructionYN",
    "Latitude", "Longitude", "OriginalListPrice", "ListPrice",
] if col in cleaned]

print("\nFractional garage / parking values — first 30 records:")
fractional_mask = pd.Series(False, index=cleaned.index)
for col in ["GarageSpaces", "ParkingTotal"]:
    if col in numeric_features:
        fractional_mask |= cleaned[col].notna() & cleaned[col].mod(1).ne(0)
display(cleaned.loc[fractional_mask, context_columns].head(30))

if "YearBuilt" in numeric_features:
    print("\nConstruction year after sale year — all flagged records:")
    display(cleaned.loc[cleaned["YearBuilt"].gt(cleaned["CloseDate"].dt.year), context_columns])

if "LotSizeSquareFeet" in numeric_features:
    print("\nSmallest positive lot measurements — compare with source lot fields:")
    small_lots = cleaned.loc[cleaned["LotSizeSquareFeet"].gt(0)].nsmallest(10, "LotSizeSquareFeet")
    display(small_lots[context_columns])
    print("\nLargest lot measurements — compare with source lot fields:")
    display(cleaned.nlargest(10, "LotSizeSquareFeet")[context_columns])

for col in ["GarageSpaces", "ParkingTotal"]:
    if col in numeric_features:
        print(f"\nLargest {col} values — first 10 records:")
        display(cleaned.nlargest(10, col)[context_columns])

# Zero coordinates or a positive longitude are inconsistent with a California
# address. Flag both coordinates together; never silently flip a sign.
if {"Latitude", "Longitude"}.issubset(numeric_features):
    coordinate_review_mask = (
        cleaned["Latitude"].eq(0) | cleaned["Longitude"].ge(0)
    )
    print("\nCalifornia coordinate review — zero latitude or nonnegative longitude:")
    print("Flagged records:", int(coordinate_review_mask.sum()))
    display(cleaned.loc[coordinate_review_mask, context_columns].head(30))

print("\nLowest 10 prices with listing-price context (context does not verify a sale price):")
display(cleaned.nsmallest(10, "ClosePrice")[context_columns])
print("\nHighest 10 prices with listing-price context (context does not verify a sale price):")
display(cleaned.nlargest(10, "ClosePrice")[context_columns])



Fractional garage / parking values — first 30 records:


,ListingKeyNumeric,source_file,CloseDate,ClosePrice,City,PostalCode,LivingArea,LotSizeSquareFeet,LotSizeAcres,LotSizeArea,GarageSpaces,ParkingTotal,YearBuilt,NewConstructionYN,Latitude,Longitude,OriginalListPrice,ListPrice
129,1146105170,CRMLSSold202511.csv,2025-11-12,1595000.0,Rancho Mirage,92270,2953.0,5227.0,0.1200,5227.00,2.5,2.5,1980.0,False,33.760494,-116.424632,1595000.0,1595000.0
1751,1139703470,CRMLSSold202511.csv,2025-11-20,690000.0,Artesia,90701,1274.0,5189.0,0.1191,5189.00,1.5,1.5,1961.0,False,33.876867,-118.087520,720000.0,720000.0
2511,1138459348,CRMLSSold202511.csv,2025-11-05,1995000.0,La Quinta,92253,3567.0,17400.0,0.3994,17400.00,2.5,2.5,2020.0,False,33.654507,-116.266497,1995000.0,1995000.0
2524,1138450957,CRMLSSold202511.csv,2025-11-24,825000.0,Rancho Mirage,92270,2603.0,4356.0,0.1000,4356.00,2.5,2.5,1996.0,False,33.768586,-116.400820,825000.0,825000.0
3804,1137011210,CRMLSSold202511.csv,2025-11-24,1975000.0,La Quinta,92253,4054.0,16500.0,0.3788,16500.00,2.5,2.5,1996.0,False,33.648663,-116.261013,1995000.0,1995000.0
4815,1135508522,CRMLSSold202511.csv,2025-11-14,1276000.0,Corona,92882,3872.0,16117.0,0.3700,16117.00,3.5,3.5,2001.0,False,33.879230,-117.642140,1289900.0,1289900.0
5582,1132943919,CRMLSSold202511.csv,2025-11-05,420000.0,Indio,92201,1627.0,5227.0,0.1200,5227.00,2.5,4.5,2002.0,False,33.693660,-116.223153,435000.0,435000.0
6280,1130094393,CRMLSSold202511.csv,2025-11-14,449000.0,Chico,95973,1439.0,6534.0,0.1500,6534.00,2.5,2.5,1998.0,False,39.766941,-121.882877,519900.0,465000.0
7535,1120225321,CRMLSSold202511.csv,2025-11-12,374000.0,Banning,92220,1505.0,4792.0,0.1100,4792.00,2.5,2.5,1988.0,False,33.919506,-116.934785,389000.0,379900.0
7554,1120173127,CRMLSSold202511.csv,2025-11-06,2350000.0,Carlsbad,92009,3641.0,20037.6,0.4600,0.46,2.5,4.5,2007.0,NaN,33.107831,-117.233730,2388800.0,2388800.0



Construction year after sale year — all flagged records:


,ListingKeyNumeric,source_file,CloseDate,ClosePrice,City,PostalCode,LivingArea,LotSizeSquareFeet,LotSizeAcres,LotSizeArea,GarageSpaces,ParkingTotal,YearBuilt,NewConstructionYN,Latitude,Longitude,OriginalListPrice,ListPrice
6195,1130409869,CRMLSSold202511.csv,2025-11-10,1488000.0,Costa Mesa,92627,4277.0,7500.0,0.1722,7500.0,3.0,3.0,2026.0,True,33.649716,-117.927420,1488000.0,1488000.0
9153,1113027527,CRMLSSold202511.csv,2025-11-21,774339.0,Moreno Valley,92557,3267.0,11836.0,0.2717,11836.0,3.0,3.0,2026.0,True,33.950022,-117.209124,794990.0,759990.0
15865,1137576367,CRMLSSold202512.csv,2025-12-18,6195000.0,Santa Monica,90403,5018.0,8004.0,0.1837,8004.0,2.0,2.0,2026.0,False,34.042267,-118.478799,6195000.0,6195000.0
18816,1119702034,CRMLSSold202512.csv,2025-12-23,748690.0,Riverside,92508,2723.0,4250.0,0.0976,4250.0,2.0,2.0,2026.0,True,33.876704,-117.330440,760595.0,729990.0
18818,1119700581,CRMLSSold202512.csv,2025-12-23,737990.0,Riverside,92508,3008.0,4250.0,0.0976,4250.0,2.0,2.0,2026.0,True,33.876713,-117.330080,781665.0,749990.0
20010,1107716688,CRMLSSold202512.csv,2025-12-12,1573813.0,Placentia,92870,2255.0,3886.0,0.0892,3886.0,2.0,2.0,2026.0,True,33.877560,-117.840558,1594000.0,1609000.0



Smallest positive lot measurements — compare with source lot fields:


...

,ListingKeyNumeric,source_file,CloseDate,ClosePrice,City,PostalCode,LivingArea,LotSizeSquareFeet,LotSizeAcres,LotSizeArea,GarageSpaces,ParkingTotal,YearBuilt,NewConstructionYN,Latitude,Longitude,OriginalListPrice,ListPrice
4365,1136698349,CRMLSSold202511.csv,2025-11-19,740000.0,Pico Rivera,90060,1382.0,6854.0,0.1573,6854.0,2.0,2.0,1957.0,False,34.018650,118.065830,660000.0,660000.0
5270,1133697040,CRMLSSold202511.csv,2025-11-07,615000.0,Santa Maria,93455,1874.0,2613.0,0.0600,2613.0,2.0,2.0,2004.0,False,34.916920,120.432670,607900.0,607900.0
6954,1126964307,CRMLSSold202511.csv,2025-11-10,648000.0,Lake Elsinore,92532,2852.0,7099.0,0.1630,7099.0,2.0,2.0,2019.0,False,33.667652,117.270769,699000.0,645000.0
7539,1120218033,CRMLSSold202511.csv,2025-11-07,545000.0,Moreno Valley,92553,1966.0,7841.0,0.1800,7841.0,2.0,2.0,1983.0,False,33.937295,117.260059,575000.0,550000.0
16455,1136784325,CRMLSSold202512.csv,2025-12-03,1532264.0,Marina,93933,3067.0,6000.0,0.1377,6000.0,3.0,3.0,2025.0,False,0.000000,0.000000,1640844.0,1520844.0
19822,1112143463,CRMLSSold202512.csv,2025-12-04,540000.0,Palmdale,93551,1912.0,4773.0,0.1096,4773.0,2.0,2.0,2006.0,False,0.000000,0.000000,579000.0,549998.0
19877,1111616733,CRMLSSold202512.csv,2025-12-23,1529494.0,Napa,94558,2852.0,5335.0,0.1225,5335.0,2.0,2.0,2025.0,False,0.000000,0.000000,1417000.0,1458500.0
22216,1147285289,CRMLSSold202601.csv,2026-01-15,425000.0,Hesperia,92344,1501.0,5000.0,0.1148,5000.0,2.0,2.0,1995.0,False,34.423064,117.366349,425000.0,425000.0
28411,1151742468,CRMLSSold202602.csv,2026-02-19,255000.0,Lancaster,93534,848.0,8898.0,0.2043,8898.0,2.0,2.0,1939.0,False,35.691123,118.140030,250500.0,250500.0
33556,1146479685,CRMLSSold202602.csv,2026-02-26,650000.0,Lancaster,93536,3215.0,7784.0,0.1787,7784.0,2.0,2.0,2008.0,False,0.000000,0.000000,649999.0,649999.0



Lowest 10 prices with listing-price context (context does not verify a sale price):


,ListingKeyNumeric,source_file,CloseDate,ClosePrice,City,PostalCode,LivingArea,LotSizeSquareFeet,LotSizeAcres,LotSizeArea,GarageSpaces,ParkingTotal,YearBuilt,NewConstructionYN,Latitude,Longitude,OriginalListPrice,ListPrice
24745,1144223437,CRMLSSold202601.csv,2026-01-30,1.75,Palm Desert,92211,3513.0,11326.0,0.2600,11326.00,3.0,3.0,2000.0,False,33.744081,-116.328238,1795000.0,1749000.0
32838,1147508351,CRMLSSold202602.csv,2026-02-04,8000.00,Trona,93562,984.0,6200.0,0.1423,6200.00,0.0,0.0,1962.0,False,35.746692,-117.399521,8000.0,8000.0
27485,1110035161,CRMLSSold202601.csv,2026-01-17,8300.00,Altadena,91001,3886.0,19206.0,0.4409,19206.00,3.0,3.0,1999.0,False,34.208003,-118.154204,2199000.0,2000000.0
36506,1108190915,CRMLSSold202602.csv,2026-02-11,18000.00,Blythe,92225,1488.0,13068.0,0.3000,13068.00,0.0,0.0,1957.0,False,33.605124,-114.729659,28000.0,22000.0
43143,1151314793,CRMLSSold202603.csv,2026-03-26,28000.00,Landers,92285,408.0,54450.0,1.2500,54450.00,0.0,5.0,1968.0,False,34.369680,-116.472823,34900.0,34900.0
7451,1120335789,CRMLSSold202511.csv,2025-11-20,29000.00,Yermo,92398,960.0,43560.0,1.0000,43560.00,2.0,2.0,1950.0,False,34.897592,-116.829557,29000.0,29000.0
102,1146281620,CRMLSSold202511.csv,2025-11-26,29900.00,Buttonwillow,93206,1198.0,3920.0,0.0900,3920.00,0.0,2.0,1953.0,False,35.400201,-119.466655,29000.0,29900.0
12175,1145717594,CRMLSSold202512.csv,2025-12-10,31000.00,Trona,93562,1942.0,20908.8,0.4800,0.48,2.0,2.0,1953.0,False,35.751441,-117.395198,40000.0,40000.0
25275,1138900104,CRMLSSold202601.csv,2026-01-27,31000.00,Bakersfield,93306,1309.0,6969.0,0.1600,6969.00,2.0,2.0,1955.0,False,35.370551,-118.948818,320000.0,310000.0
18299,1126691917,CRMLSSold202512.csv,2025-12-08,37000.00,Coarsegold,93614,1456.0,58370.4,1.3400,1.34,2.0,2.0,1992.0,False,37.264099,-119.692443,385000.0,385000.0



Highest 10 prices with listing-price context (context does not verify a sale price):


,ListingKeyNumeric,source_file,CloseDate,ClosePrice,City,PostalCode,LivingArea,LotSizeSquareFeet,LotSizeAcres,LotSizeArea,GarageSpaces,ParkingTotal,YearBuilt,NewConstructionYN,Latitude,Longitude,OriginalListPrice,ListPrice
58811,1149697906,CRMLSSold202604.csv,2026-04-27,796000000.0,Valley Center,92082,1936.0,14810.40,0.3400,0.34,3.0,3.0,1977.0,NaN,33.250628,-117.135871,849900.0,824900.0
58371,1150712418,CRMLSSold202604.csv,2026-04-09,768500000.0,San Diego,92154,1444.0,0.12,0.0000,0.12,2.0,4.0,1978.0,NaN,32.572879,-117.062559,750000.0,750000.0
7186,1126046881,CRMLSSold202511.csv,2025-11-24,740000000.0,San Diego,92154,1271.0,5100.00,0.1171,5100.00,2.0,2.0,1973.0,NaN,32.566454,-117.095029,750000.0,735000.0
45337,1148201765,CRMLSSold202603.csv,2026-03-26,664250000.0,Chula Vista,91911,1992.0,0.23,0.0000,0.23,2.0,4.0,1959.0,False,32.625679,-117.065524,879000.0,750000.0
55080,1153305774,CRMLSSold202604.csv,2026-04-22,645000000.0,Oceanside,92056,1104.0,4300.00,0.0987,4300.00,1.0,2.0,1972.0,NaN,33.185572,-117.302394,645000.0,645000.0
25905,1135757473,CRMLSSold202601.csv,2026-01-19,472000000.0,Victorville,92395,1910.0,8200.00,0.1882,8200.00,2.0,7.0,2000.0,NaN,34.490545,-117.257502,489000.0,479000.0
9399,1111473680,CRMLSSold202511.csv,2025-11-14,199947850.0,Topanga,90290,1704.0,74393.00,1.7078,74393.00,2.0,10.0,1964.0,False,34.123623,-118.599896,1499000.0,1499000.0
50445,1155438031,CRMLSSold202604.csv,2026-04-30,60000000.0,Montecito,93108,15006.0,261796.00,6.0100,261796.00,NaN,10.0,NaN,False,34.430419,-119.646132,65000000.0,65000000.0
36733,1155445977,CRMLSSold202603.csv,2026-03-16,47000000.0,Beverly Hills,90210,10286.0,52591.00,1.2073,52591.00,NaN,0.0,1937.0,False,34.090074,-118.407828,59500000.0,59500000.0
48359,1157576793,CRMLSSold202604.csv,2026-04-14,41250000.0,Beverly Hills,90210,NaN,39924.00,0.9165,39924.00,NaN,0.0,1928.0,False,34.082565,-118.420343,41250000.0,41250000.0


### Apply measurement-validity rules and provisional screening

Treat nonpositive areas and negative counts as missing. Also treat the following unverified measurements as missing while preserving the property rows and original measurements in the audit:

- Positive lots smaller than 1 square foot.
- Suspected double conversion: source `LotSizeArea` is at least 1,000, approximately equals `LotSizeAcres`, and `LotSizeSquareFeet` approximately equals that value multiplied by 43,560. No corrected value is inferred.
- Garage values above 20 or total parking above 50. These are conservative baseline screening assumptions, not physical limits or CRMLS standards. Valid large-capacity properties may be affected; review the audit and reconsider these assumptions when documentation becomes available.

Retain zeros in count fields, fractional garage/parking values, and the reviewed future construction years. Treat both coordinates as missing when latitude is zero or longitude is nonnegative. Other location discrepancies remain for review. No rounding, inferred price correction, or coordinate sign correction is applied.


In [77]:
# Apply only established validity rules; keep a record of every modified field.
measurement_change_frames = []
measurement_rule_counts = []
for col in numeric_features:
    if col in {"LivingArea", "LotSizeSquareFeet"}:
        invalid = cleaned[col].le(0)
        reason = "Nonpositive area treated as missing"
    elif col in {"BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal"}:
        invalid = cleaned[col].lt(0)
        reason = "Negative count treated as missing"
    else:
        continue
    measurement_rule_counts.append({"Feature": col, "Rule": reason,
                                    "Values changed": int(invalid.sum())})
    if invalid.any():
        changes = cleaned.loc[invalid, ["ListingKeyNumeric", "source_file", col]].copy()
        changes = changes.rename(columns={col: "original_value"})
        changes["feature"] = col
        changes["rule"] = reason
        measurement_change_frames.append(changes)
        cleaned.loc[invalid, col] = np.nan
# Screen unverified positive measurements without inventing replacements.
# Limits are explicit baseline assumptions; large valid properties may be affected.
positive_review_masks = {}
if "LotSizeSquareFeet" in numeric_features:
    lot = cleaned["LotSizeSquareFeet"]
    positive_review_masks[("LotSizeSquareFeet", "Positive lot below 1 square foot; unit unverified")] = (
        lot.gt(0) & lot.lt(MEASUREMENT_SCREENING["minimum_positive_lot_sqft"])
    )
    if {"LotSizeArea", "LotSizeAcres"}.issubset(cleaned.columns):
        source_area = pd.to_numeric(cleaned["LotSizeArea"], errors="coerce")
        source_acres = pd.to_numeric(cleaned["LotSizeAcres"], errors="coerce")
        suspected_conversion = (
            source_area.ge(MEASUREMENT_SCREENING["suspected_double_conversion_minimum_area"])
            & np.isfinite(source_area) & np.isfinite(source_acres) & np.isfinite(lot)
            & np.isclose(source_area, source_acres, rtol=1e-6, atol=0.01)
            & np.isclose(lot, source_area * SQFT_PER_ACRE, rtol=1e-6, atol=1.0)
        )
        positive_review_masks[("LotSizeSquareFeet", "Source area/acres repeat and appear converted again; units unverified")] = suspected_conversion
for col, limit_key in [("GarageSpaces", "garage_review_limit"), ("ParkingTotal", "parking_review_limit")]:
    if col in numeric_features:
        limit = MEASUREMENT_SCREENING[limit_key]
        positive_review_masks[(col, f"Above provisional review limit of {limit:g}; value unverified")] = cleaned[col].gt(limit)
for (col, reason), flagged in positive_review_masks.items():
    # Count changes only where the measurement is still present.
    changed = flagged & cleaned[col].notna()
    measurement_rule_counts.append({"Feature": col, "Rule": reason,
                                    "Values changed": int(changed.sum())})
    if changed.any():
        changes = cleaned.loc[changed, ["ListingKeyNumeric", "source_file", col]].copy()
        changes = changes.rename(columns={col: "original_value"})
        changes["feature"] = col
        changes["rule"] = reason
        measurement_change_frames.append(changes)
        cleaned.loc[changed, col] = np.nan

# Invalidate a demonstrably incorrect coordinate pair together. A sign change
# could still point to the wrong city, so do not infer replacement coordinates.
if {"Latitude", "Longitude"}.issubset(numeric_features):
    invalid_pair = cleaned["Latitude"].eq(0) | cleaned["Longitude"].ge(0)
    print("Invalid California coordinate pairs treated as missing:", int(invalid_pair.sum()))
    for col in ["Latitude", "Longitude"]:
        changed = invalid_pair & cleaned[col].notna()
        reason = "Zero latitude or nonnegative longitude invalidates California coordinate pair"
        measurement_rule_counts.append({"Feature": col, "Rule": reason,
                                        "Values changed": int(changed.sum())})
        if changed.any():
            changes = cleaned.loc[changed, ["ListingKeyNumeric", "source_file", col]].copy()
            changes = changes.rename(columns={col: "original_value"})
            changes["feature"] = col
            changes["rule"] = reason
            measurement_change_frames.append(changes)
        cleaned.loc[invalid_pair, col] = np.nan

measurement_changes = (
    pd.concat(measurement_change_frames, ignore_index=True)
    if measurement_change_frames else pd.DataFrame(columns=[
        "ListingKeyNumeric", "source_file", "original_value", "feature", "rule"
    ])
)
print("\nApplied measurement rules — field changes, not row removals:")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(measurement_rule_counts))
print("Property rows retained:", len(cleaned))

# Refresh missingness after the newly invalidated measurements.
feature_missingness = pd.DataFrame({
    "Missing rows": cleaned[feature_columns].isna().sum(),
    "Available rows": cleaned[feature_columns].notna().sum(),
    "Missing (%)": cleaned[feature_columns].isna().mean().mul(100),
}).sort_values("Missing (%)", ascending=False, kind="stable")
feature_missingness.index.name = "Feature"
print("\nFeature missingness after measurement cleaning:")
display(feature_missingness.style.format({"Missing rows": "{:,.0f}",
                                         "Available rows": "{:,.0f}",
                                         "Missing (%)": "{:.2f}%"}))


Invalid California coordinate pairs treated as missing: 17

Applied measurement rules — field changes, not row removals:


,Feature,Rule,Values changed
0,LivingArea,Nonpositive area treated as missing,18
1,LotSizeSquareFeet,Nonpositive area treated as missing,43
2,BedroomsTotal,Negative count treated as missing,0
3,BathroomsTotalInteger,Negative count treated as missing,0
4,GarageSpaces,Negative count treated as missing,0
5,ParkingTotal,Negative count treated as missing,8
6,LotSizeSquareFeet,Positive lot below 1 square foot; unit unverified,138
7,LotSizeSquareFeet,Source area/acres repeat and appear converted again; units unverified,56
8,GarageSpaces,Above provisional review limit of 20; value unverified,9
9,ParkingTotal,Above provisional review limit of 50; value unverified,29


Property rows retained: 60615

Feature missingness after measurement cleaning:


,Missing rows,Available rows,Missing (%)
Feature,,,
Stories,"6,230","54,385",10.28%
GarageSpaces,"2,350","58,265",3.88%
LotSizeSquareFeet,"1,293","59,322",2.13%
Latitude,230,"60,385",0.38%
Longitude,230,"60,385",0.38%
LivingArea,47,"60,568",0.08%
ParkingTotal,37,"60,578",0.06%
YearBuilt,37,"60,578",0.06%
City,7,"60,608",0.01%


### Set aside manually reviewed sale-price anomalies

Exclude the 11 reviewed listing keys from the baseline cleaned table and keep their records in `suspicious_sale_prices_for_review.csv`. These are record-specific review decisions, not a universal price threshold or a rule fitted on held-out model performance. Do not replace sale prices with listing prices or infer a scaling correction. The six low-price and three high-price records outside this review list in the displayed extremes remain provisionally retained.

This review included records from the eventual test month. Document these cleaning decisions before evaluation and freeze the rules; the holdout is reserved from model tuning, but was not hidden from data-quality inspection. The review list is specific to the current exports and needs reassessment for new inputs.


In [78]:
# Exclude only explicitly reviewed records; retain full originals in raw.
price_review_mask = cleaned["ListingKeyNumeric"].isin(PRICE_REVIEW_REASONS)
price_review = cleaned.loc[price_review_mask].copy()
price_review["review_reason"] = price_review["ListingKeyNumeric"].map(PRICE_REVIEW_REASONS)
before = len(cleaned)
cleaned = cleaned.loc[~price_review_mask].copy()
audit_step("manually reviewed suspicious sale prices set aside", before, len(cleaned))
print(f"Set aside {len(price_review):,} suspicious-price records.")
print("Reviewed keys absent from the current cleaned table:",
      sorted(set(PRICE_REVIEW_REASONS) - set(price_review["ListingKeyNumeric"])))
assert not cleaned.empty, "No records remain after price review."
assert not cleaned["ListingKeyNumeric"].isin(PRICE_REVIEW_REASONS).any()
print("\nUpdated row audit after price review:")
display(pd.DataFrame(row_audit))
print("\nRetained close-price summary after excluding reviewed anomalies:")
display(cleaned["ClosePrice"].describe(percentiles=[0.01, 0.5, 0.99]).to_frame())


Set aside 11 suspicious-price records.
Reviewed keys absent from the current cleaned table: []

Updated row audit after price review:


,step,rows_remaining,rows_removed
0,imported,122756,0
1,California residential single-family scope,60694,62062
2,"valid price, date, and listing key",60694,0
3,exact duplicates,60694,0
4,conflicting listing keys set aside for review,60615,79
5,manually reviewed suspicious sale prices set a...,60604,11



Retained close-price summary after excluding reviewed anomalies:


,ClosePrice
count,6.060400e+04
mean,1.250487e+06
std,1.426157e+06
min,8.000000e+03
1%,2.300000e+05
50%,8.800000e+05
99%,6.500000e+06
max,6.000000e+07


### Section 2 decisions and limitations

Retain all current property/location candidates. Convert invalid or explicitly screened measurements to missing instead of dropping property rows. Preserve original changed values in `measurement_cleaning_audit.csv`. Keep fractional garage/parking values without rounding: the supplied documentation does not establish their reporting convention. Retain the reviewed future construction years provisionally, including one record without supporting new-construction status.

Suspected lot-unit errors are not repaired through guessed conversions. Provisional garage/parking screening may mask valid unusually large facilities. The checks do not detect every unit error or validate every coordinate against a California boundary or address. Manually reviewed price exclusions are specific to this dataset. These uncertainties, feature availability for off-market homes, and training only on sold properties limit what the model can establish about any California address.

Missing features remain in the human-readable cleaned CSV. Numeric median imputation, missing indicators, categorical encoding, and scaling are fitted only on each training fold. The following output refreshes row counts, missingness, and field-change counts after all section 2 decisions. It reflects a baseline cleaning policy, not verified accuracy of every source record.


In [79]:
# Final pre-export summary: use current rows after price review, avoiding stale counts.
feature_missingness = pd.DataFrame({
    "Missing rows": cleaned[feature_columns].isna().sum(),
    "Available rows": cleaned[feature_columns].notna().sum(),
    "Missing (%)": cleaned[feature_columns].isna().mean().mul(100),
}).sort_values("Missing (%)", ascending=False, kind="stable")
feature_missingness.index.name = "Feature"
print(f"\nSection 2 complete — {len(cleaned):,} property rows available for baseline export.")
print("Fractional garage/parking values retained; no rounding or inferred unit corrections.")
print("\nFinal feature missingness after measurement cleaning and price review:")
display(feature_missingness.style.format({"Missing rows": "{:,.0f}",
                                         "Available rows": "{:,.0f}",
                                         "Missing (%)": "{:.2f}%"}))
print("\nMeasurement changes — counts include changes to rows later set aside for price review:")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(measurement_rule_counts))

# Guard the explicit rules before writing files; these checks do not verify source truth.
for col in ["LivingArea", "LotSizeSquareFeet"]:
    if col in numeric_features:
        assert cleaned[col].dropna().gt(0).all(), f"Nonpositive {col} remains."
for col in ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal"]:
    if col in numeric_features:
        assert cleaned[col].dropna().ge(0).all(), f"Negative {col} remains."
for col, limit_key in [("GarageSpaces", "garage_review_limit"), ("ParkingTotal", "parking_review_limit")]:
    if col in numeric_features:
        assert cleaned[col].dropna().le(MEASUREMENT_SCREENING[limit_key]).all()
if "LotSizeSquareFeet" in numeric_features:
    assert cleaned["LotSizeSquareFeet"].dropna().ge(MEASUREMENT_SCREENING["minimum_positive_lot_sqft"]).all()



Section 2 complete — 60,604 property rows available for baseline export.
Fractional garage/parking values retained; no rounding or inferred unit corrections.

Final feature missingness after measurement cleaning and price review:


,Missing rows,Available rows,Missing (%)
Feature,,,
Stories,"6,229","54,375",10.28%
GarageSpaces,"2,350","58,254",3.88%
LotSizeSquareFeet,"1,291","59,313",2.13%
Latitude,230,"60,374",0.38%
Longitude,230,"60,374",0.38%
LivingArea,47,"60,557",0.08%
ParkingTotal,37,"60,567",0.06%
YearBuilt,37,"60,567",0.06%
City,7,"60,597",0.01%



Measurement changes — counts include changes to rows later set aside for price review:


,Feature,Rule,Values changed
0,LivingArea,Nonpositive area treated as missing,18
1,LotSizeSquareFeet,Nonpositive area treated as missing,43
2,BedroomsTotal,Negative count treated as missing,0
3,BathroomsTotalInteger,Negative count treated as missing,0
4,GarageSpaces,Negative count treated as missing,0
5,ParkingTotal,Negative count treated as missing,8
6,LotSizeSquareFeet,Positive lot below 1 square foot; unit unverified,138
7,LotSizeSquareFeet,Source area/acres repeat and appear converted again; units unverified,56
8,GarageSpaces,Above provisional review limit of 20; value unverified,9
9,ParkingTotal,Above provisional review limit of 50; value unverified,29


## Cleaned CSV export

Run after reviewing the scope audit, duplicate checks, and feature missingness above. Export property/location candidates, target, sale date, and audit identifiers; omit agent/contact fields and other unused raw columns. Missing features are intentional and are imputed only within training folds. No percentile clipping is applied. This table is a baseline cleaning deliverable; positive price extremes and geographic discrepancies still require review.

The later modeling cells create a separate file with selected training-window membership. They do not overwrite this cleaned table.

Also save `conflicting_listing_records_for_review.csv` and `preprocessing_metadata.json` with content fingerprints, package versions, cleaning rules, and row counts. Outputs are replaced when this cell is rerun. Reproduce a run using identical inputs, notebook code, and recorded package versions.


In [80]:
audit_columns = ["ListingKeyNumeric", "StateOrProvince", "CloseDate", "source_file"]
export_columns = audit_columns + ["ClosePrice"] + feature_columns
cleaned_export = cleaned[export_columns].copy()
assert not cleaned_export.empty
assert cleaned_export["ListingKeyNumeric"].is_unique
assert cleaned_export["StateOrProvince"].eq("CA").all()
assert (cleaned_export["ClosePrice"].gt(0) & np.isfinite(cleaned_export["ClosePrice"])).all()
assert cleaned_export["CloseDate"].notna().all()
cleaned_csv_path = data_dir / "cleaned_single_family_sales.csv"
# Write stable row order and date formatting for reproducible comparisons.
cleaned_export = cleaned_export.sort_values(
    ["CloseDate", "ListingKeyNumeric"], kind="stable"
)
cleaned_export.to_csv(cleaned_csv_path, index=False, date_format="%Y-%m-%d")

# Review artifacts exclude contact details; the full conflicts remain in memory.
review_columns = [
    col for col in [*audit_columns, "PropertyType", "PropertySubType", "ClosePrice",
                   *feature_columns, "review_reason"] if col in duplicate_review
]
duplicate_review_path = data_dir / "conflicting_listing_records_for_review.csv"
duplicate_review[review_columns].sort_values(
    ["ListingKeyNumeric", "source_file"], kind="stable"
).to_csv(duplicate_review_path, index=False, date_format="%Y-%m-%d")

# Preserve manually reviewed price anomalies without exposing contact fields.
price_review_path = data_dir / "suspicious_sale_prices_for_review.csv"
price_review_columns = [
    col for col in [*audit_columns, "ClosePrice", *feature_columns,
                   "OriginalListPrice", "ListPrice", "review_reason"] if col in price_review
]
price_review[price_review_columns].sort_values(
    ["ListingKeyNumeric", "source_file"], kind="stable"
).to_csv(price_review_path, index=False, date_format="%Y-%m-%d")

# Persist original values for measurement changes separately from model inputs.
measurement_changes_path = data_dir / "measurement_cleaning_audit.csv"
measurement_changes.to_csv(measurement_changes_path, index=False)

# Record exact input contents, software versions, policies, and output contents.
run_metadata = {
    "python_version": platform.python_version(),
    "package_versions": {name: version(name) for name in ["numpy", "pandas", "scikit-learn"]},
    "inputs": input_manifest,
    "notebook_sha256": file_sha256(project_root / "notebooks" / "02_preprocessing.ipynb"),
    "scope": {"state": "CA", "property_type": "Residential", "property_subtype": "SingleFamilyResidence"},
    "duplicate_policy": DUPLICATE_POLICY,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "coordinate_bounds": COORDINATE_BOUNDS,
    "missing_features": "Retained in CSV; imputation fitted on each training fold only",
    "price_rule": "Finite and positive; manually reviewed suspicious keys set aside",
    "price_review_reasons": PRICE_REVIEW_REASONS,
    "price_review_records": len(price_review),
    "price_review_sha256": file_sha256(price_review_path),
    "row_audit": row_audit,
    "review_records": len(duplicate_review),
    "measurement_rules": measurement_rule_counts,
    "measurement_screening_assumptions": MEASUREMENT_SCREENING,
    "fractional_parking_policy": "Retain fractional values; reporting convention is undocumented",
    "lot_unit_policy": "Set aside suspect measurements; no inferred conversions",
    "future_construction_policy": "Retain reviewed future years provisionally",
    "measurement_audit_sha256": file_sha256(measurement_changes_path),
    "unresolved_reviews": [
        "Additional price anomalies beyond reviewed examples", "Unit issues and extreme measurements outside explicit screens",
        "Other coordinate/city inconsistencies beyond zero/sign checks", "Prediction-time feature availability",
    ],
    "cleaned_csv_sha256": file_sha256(cleaned_csv_path),
    "duplicate_review_sha256": file_sha256(duplicate_review_path),
    "model_configuration": {
        "candidate_windows": CANDIDATE_WINDOWS, "validation_offsets": VALIDATION_OFFSETS,
        "ridge_alpha": RIDGE_ALPHA, "ridge_solver": "lsqr",
    },
}
metadata_path = data_dir / "preprocessing_metadata.json"
metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n", encoding="utf-8")
print(f"Exported {len(cleaned_export):,} rows and {len(export_columns)} columns to {cleaned_csv_path}")
display(pd.DataFrame(row_audit))

print(f"Duplicate review records: {len(duplicate_review):,}; saved to {duplicate_review_path}")
print(f"Run metadata saved to {metadata_path}")

print(f"Suspicious price records: {len(price_review):,}; saved to {price_review_path}")


Exported 60,604 rows and 17 columns to /Users/daira/IDX-Exchange-Data-Science/data/cleaned_single_family_sales.csv


,step,rows_remaining,rows_removed
0,imported,122756,0
1,California residential single-family scope,60694,62062
2,"valid price, date, and listing key",60694,0
3,exact duplicates,60694,0
4,conflicting listing keys set aside for review,60615,79
5,manually reviewed suspicious sale prices set a...,60604,11


Duplicate review records: 79; saved to /Users/daira/IDX-Exchange-Data-Science/data/conflicting_listing_records_for_review.csv
Run metadata saved to /Users/daira/IDX-Exchange-Data-Science/data/preprocessing_metadata.json
Suspicious price records: 11; saved to /Users/daira/IDX-Exchange-Data-Science/data/suspicious_sale_prices_for_review.csv


## 3. Reserve the latest month; inspect monthly coverage

A window uses calendar months immediately preceding its evaluation month. Missing months are reported and windows with gaps are skipped. A latest month with few records may be incomplete; document this, but retain the required latest-month test split.


In [81]:
sale_month = cleaned["CloseDate"].dt.to_period("M")
test_month = sale_month.max()
# Show zero-count months explicitly. Nonzero counts do not prove a full month's data.
expected_months = pd.period_range(sale_month.min(), test_month, freq="M")
monthly_counts = sale_month.value_counts().reindex(expected_months, fill_value=0).sort_index()
print("\nMonthly sale coverage, including gaps:")
display(monthly_counts.rename("rows").to_frame())
print("Reserved test month:", test_month)
print("Latest observed sale date:", cleaned["CloseDate"].max())

def window_mask(end_month, months):
    """Select the calendar months immediately preceding an evaluation month."""
    return sale_month.ge(end_month - months) & sale_month.lt(end_month)

def has_monthly_coverage(end_month, months):
    """Check for at least one sale in each month; this does not prove complete exports."""
    expected = pd.period_range(end_month - months, end_month - 1, freq="M")
    return monthly_counts.reindex(expected, fill_value=0).gt(0).all()

print("Months with no retained sales:", [str(month) for month in monthly_counts.index[monthly_counts.eq(0)]])
print("Latest-month completeness needs confirmation from the data provider.")



Monthly sale coverage, including gaps:


,rows
2025-11,9718
2025-12,10432
2026-01,7473
2026-02,8950
2026-03,11551
2026-04,12480


Reserved test month: 2026-04
Latest observed sale date: 2026-04-30 00:00:00
Months with no retained sales: []
Latest-month completeness needs confirmation from the data provider.


## 4. Tune X on shared February and March validation months

Compare candidate training windows on the same **February and March 2026** validation months, then select the lowest mean monthly validation MAE. April 2026 remains reserved for final testing. Each training window uses only the calendar months preceding its validation month.

**Adjustment for available history:** the exports begin in November 2025. Including January as a validation month would make a three-month training window ineligible because October is unavailable. Using February and March provides enough history to compare **1- and 3-month windows** fairly:

| Validation month | 1-month training window | 3-month training window |
| --- | --- | --- |
| February 2026 | January 2026 | November 2025–January 2026 |
| March 2026 | February 2026 | December 2025–February 2026 |

Six- and twelve-month candidates remain in the configuration but are skipped for insufficient history. This adjustment is based on coverage, before inspecting validation performance. Only two validation months are available for this comparison, so the selected window may be sensitive to those months. For future exports, reassess the offsets and available history.

Imputation, encoding, scaling, and empty-column selection are fitted only on each training fold. Missing indicators and unknown-category handling support later data. Ridge with `lsqr` and chronological folds is deterministic. This selects a training window for the Ridge baseline; other models may favor different windows.


In [82]:
def make_model(train_frame):
    """Build a deterministic Ridge baseline using features present in this training fold."""
    num = [c for c in numeric_features if train_frame[c].notna().any()]
    cat = [c for c in categorical_features if train_frame[c].notna().any()]
    transformers = []
    if num:
        transformers.append(("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
        ]), num))
    if cat:
        transformers.append(("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), cat))
    if not transformers:
        raise ValueError("No non-empty training features.")
    return Pipeline([("preprocess", ColumnTransformer(transformers)),
                     ("model", Ridge(alpha=RIDGE_ALPHA, solver="lsqr"))])

candidate_windows = CANDIDATE_WINDOWS
validation_months = [test_month - offset for offset in VALIDATION_OFFSETS
                     if (sale_month == test_month - offset).any()]
print("Shared validation months:", [str(month) for month in validation_months])
# Compare candidates on the same folds to avoid comparing different time periods.
eligible_windows = [x for x in candidate_windows if validation_months and
                    all(has_monthly_coverage(month, x) for month in validation_months)
                    and has_monthly_coverage(test_month, x)]
print("Eligible training windows (months):", eligible_windows)
print("Candidate windows lacking sufficient coverage:", sorted(set(candidate_windows) - set(eligible_windows)))
results = []
for months in eligible_windows:
    for month in validation_months:
        train_mask = window_mask(month, months)
        val_mask = sale_month.eq(month)
        model = make_model(cleaned.loc[train_mask])
        model.fit(cleaned.loc[train_mask, feature_columns], cleaned.loc[train_mask, "ClosePrice"])
        predictions = model.predict(cleaned.loc[val_mask, feature_columns])
        results.append({"training_months": months, "validation_month": str(month),
                        "train_rows": int(train_mask.sum()),
                        "validation_mae": mean_absolute_error(cleaned.loc[val_mask, "ClosePrice"], predictions)})
if not results:
    raise ValueError("Insufficient monthly coverage for these windows. Review smaller X values or validation folds.")
window_results = pd.DataFrame(results)
window_scores = window_results.groupby("training_months")["validation_mae"].mean().sort_values(kind="stable")
print("\nValidation results by training window and evaluation month:")
display(window_results)
print("\nMean validation MAE by training window (ties prefer the shorter window):")
display(window_scores.rename("mean_validation_mae").to_frame())
best_x = int(window_scores.index[0])
print("Selected training window:", best_x, "months")


Shared validation months: ['2026-02', '2026-03']
Eligible training windows (months): [1, 3]
Candidate windows lacking sufficient coverage: [6, 12]

Validation results by training window and evaluation month:


,training_months,validation_month,train_rows,validation_mae
0,1,2026-02,7473,360167.828146
1,1,2026-03,8950,349964.693284
2,3,2026-02,27623,331373.346485
3,3,2026-03,26855,331834.022406



Mean validation MAE by training window (ties prefer the shorter window):


,mean_validation_mae
training_months,
3,331603.684445
1,355066.260715


Selected training window: 3 months


## 5. Fit the selected pipeline and evaluate once on the test month

Keep the test month untouched until the window and preprocessing choices are finalized. If more changes are made after inspecting its performance, that month is no longer an untouched final test.


In [83]:
train_mask = window_mask(test_month, best_x)
test_mask = sale_month.eq(test_month)
train = cleaned.loc[train_mask].copy()
test = cleaned.loc[test_mask].copy()
assert not train.empty and not test.empty
assert set(train["ListingKeyNumeric"]).isdisjoint(test["ListingKeyNumeric"])
assert train["CloseDate"].max() < test["CloseDate"].min()
final_pipeline = make_model(train)
final_pipeline.fit(train[feature_columns], train["ClosePrice"])
print(f"Training rows: {len(train):,}; test rows: {len(test):,}")
# Run only after preprocessing decisions are finalized:
# test_predictions = final_pipeline.predict(test[feature_columns])
# print("Test MAE:", mean_absolute_error(test["ClosePrice"], test_predictions))


Training rows: 27,974; test rows: 12,480


## 6. Deliverable export — run after review

Export a cleaned, scoped, deduplicated table with original units, audit columns, and train/test membership. Remaining missing features are intentional: the fitted pipeline handles imputation, encoding, and normalization without leaking test information. Rows before the selected training window are labeled `outside_training_window`.

If the deliverable requires a fully encoded/imputed CSV, additionally export train/test model matrices using the training-fitted pipeline and document that they differ from the human-readable cleaned table. Preserve the fitted pipeline for reproducibility.


In [84]:
export_frame = cleaned_export.copy()
export_frame["split"] = "outside_training_window"
export_frame.loc[train_mask, "split"] = "train"
export_frame.loc[test_mask, "split"] = "test"
export_frame["selected_training_months"] = best_x
assert export_frame["ClosePrice"].gt(0).all()
assert export_frame["ListingKeyNumeric"].is_unique
print("Proposed export:", data_dir / "cleaned_single_family_sales_with_splits.csv")
display(export_frame["split"].value_counts().rename("rows").to_frame())
# Run after selecting and reviewing the training window:
export_frame = export_frame.sort_values(["CloseDate", "ListingKeyNumeric"], kind="stable")
export_frame.to_csv(
    data_dir / "cleaned_single_family_sales_with_splits.csv", index=False, date_format="%Y-%m-%d"
)
# Extend metadata once temporal selection has run; the cleaned export works independently.
run_metadata["temporal_selection"] = {
    "test_month": str(test_month), "selected_training_months": best_x,
    "validation_results": window_results.to_dict(orient="records"),
    "train_rows": len(train), "test_rows": len(test),
}
metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n", encoding="utf-8")
# import joblib
# joblib.dump(final_pipeline, project_root / "notebooks" / "preprocessing_pipeline.joblib")


Proposed export: /Users/daira/IDX-Exchange-Data-Science/data/cleaned_single_family_sales_with_splits.csv


,rows
split,
train,27974
outside_training_window,20150
test,12480


7381

## Key takeaways

The baseline preprocessing deliverable was exported successfully: **60,604 California residential single-family sales** from November 2025 through April 2026. The cleaned CSV contains **17 columns**, including target and audit fields; these are not all model predictors. Original units and remaining missing features are retained. Training-fitted preprocessing handles imputation, encoding, and scaling.

The split-labeled CSV contains the same records plus `split` and `selected_training_months`:

| Split | Sale months | Rows |
| --- | --- | ---: |
| Train | January–March 2026 | 27,974 |
| Test | April 2026 | 12,480 |
| Outside selected training window | November–December 2025 | 20,150 |
| Total | November 2025–April 2026 | 60,604 |

Earlier rows remain in the export for traceability and were used in some validation folds. They are excluded from the final three-month model fit; they are not rejected as poor-quality records.

### Baseline results

Shared February/March validation folds selected a **three-month training window** for Ridge. Mean monthly validation MAE was **$331,603.68**, compared with **$355,066.26** for one month, an improvement of approximately **6.6%**. Six- and twelve-month windows were ineligible because the source history was too short.

The fitted January–March model achieved **April test MAE of $354,694.32** across **12,480 sales**, approximately 7% higher than validation MAE. MAE is average absolute dollar error, not an error bound for each home. These results establish a baseline; they do not establish reliable accuracy for every California address.

### Cleaning decisions and artifacts

- Scope filtering retained 60,694 California residential single-family records from 122,756 imported records.
- Set aside all 79 records belonging to 39 conflicting listing keys, and 11 individually reviewed suspicious-price records. No sale-price replacements were inferred.
- Converted nonpositive areas, negative counts, the 17 observed invalid coordinate pairs, and measurements failing explicit provisional screens to missing. Property rows were retained through measurement cleaning.
- Preserved fractional garage/parking values without rounding and the reviewed future construction years provisionally.
- Exported `data/cleaned_single_family_sales.csv`, `data/cleaned_single_family_sales_with_splits.csv`, duplicate and price review CSVs, `data/measurement_cleaning_audit.csv`, and `data/preprocessing_metadata.json`. Keep the audits and metadata with the deliverable. The fitted pipeline export remains optional/commented in the notebook; CSVs alone do not preserve learned medians, category mappings, or model coefficients.

### Notes for the team

1. **Prediction-time inputs:** establish how property characteristics and location will be obtained for an off-market address. Sold-listing values are not proof that those same values were available at an earlier query time. Listing prices used during quality review remain excluded from predictors.
2. **Source documentation:** ask the data provider to clarify repeated listing keys, fractional parking conventions, lot units, suspicious sale prices, and whether monthly exports are complete. Restore quarantined records or measurements only after verification.
3. **Provisional screens:** garage values above 20, parking above 50, positive lots below 1 square foot, and the documented suspected double-conversion pattern were treated as unverified. These are baseline assumptions, not physical limits. They can mask legitimate measurements and do not identify all source errors. Future construction years also remain uncertain.
4. **Location validation:** zero/sign checks do not confirm California boundary membership or agreement with city/ZIP/address. Some plausible-looking coordinates may still be wrong.
5. **Evaluation limits:** only two months informed window selection. April was excluded from model tuning, but April records were inspected for quality and its model score has now been viewed. Use earlier rolling validation for further development; obtain a new untouched holdout for a subsequent final claim.
6. **Generalization:** the evaluated population consists of sold homes. Accuracy on off-market homes, future months, different geographic areas, and price segments remains unverified. Consider median absolute error and errors by price/location during future validation to complement overall MAE.
7. **Reproducibility:** save the notebook and preserve exact source files, hashes, package versions, configuration, and audits. If handing off predictions, persist the fitted pipeline as well. The current temporal metadata records the window/fold results and split counts; the reported test MAE is documented here but is not automatically stored by the existing export cell. Do not rerun and retune against April's score.
